## Load useful libraries

In [ ]:
import librosa
import multiprocessing
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

## User settings

In [ ]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 100
spark_memory = '70G'

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

In [ ]:
pdf_library_paths = pd.read_parquet(path_library_parquet)

In [ ]:
pdf_library_paths.head(5)

In [ ]:
def process_octave(y, octave_number, sr = 22050, hop_length = 512 * 200, lowest_pitch = 16.35):  # get a more precise number for C0
    chromagram = librosa.feature.chroma_cens(
        y = y,
        sr = sr,
        fmin = lowest_pitch * (octave_number + 1),
        n_octaves = 1,
        hop_length = hop_length,
    )
    return chromagram

In [ ]:
def process_song_from_song_library(filename, track_id, sampling_rate, hop_length):
    y_tick_labels = librosa.key_to_notes('C:major')  # we compute this every time for now, neither efficient nor too costly

    # there is a file not found error I need to debug later
    try:
        y, sr = librosa.load(filename, sr = sr, mono = True)
    except:
        return None
    
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    results_list = []
    for octave_number in range(0, 9):
        chromagram = process_octave(y_harmonic, octave_number, sr = sampling_rate, hop_length = hop_length)
        df = pd.DataFrame(chromagram.T)
        df.columns = [x + str(octave_number) for x in y_tick_labels]
        results_list.append(df)
    df_all_octaves = pd.concat(results_list, axis = 1)
    df_all_octaves['id'] = track_id
    return df_all_octaves

In [ ]:
%%capture

args_list = []
for i, row in pdf_library_paths.iterrows():
    args_list.append((row['path'], row['id'], sampling_rate, hop_length))

with multiprocessing.Pool(processes = 50) as pool:
    results = pool.starmap(process_song_from_song_library, args_list)

In [ ]:
pdf_library = pd.concat(results, ignore_index = True)

In [ ]:
pdf_library.head(3)

In [ ]:
len(pdf_library.index)

In [ ]:
len(pdf_library.dropna().index)

In [ ]:
len(pdf_library['id'].unique())

In [ ]:
pdf_library.dropna(inplace = True)

In [ ]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

In [ ]:
sdf_library = (
    spark
    .createDataFrame(pdf_library)
    .orderBy('id')
    .repartition('id')
    .withColumn('array_library', F.array(*value_columns))
    .select('array_library', 'id')
)

In [ ]:
sdf_library.show(5)

In [ ]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library.write.mode('overwrite').parquet(path_library_output)

In [ ]:
spark.stop()